# Skandamata: which defences stop what the attacker did?

Each behaviour is labelled with every ATT&CK mitigation linked to its technique (2.8 on average). Rank the 44 mitigations; the split is by source, so test behaviours describe unseen attackers.

Skandamata is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import pandas as pd, numpy as np
b = pd.read_csv(find("behaviours.csv")); mit = pd.read_csv(find("mitigations.csv"))
ids = list(mit["mitigation_id"])
Y = np.array([[m in set(str(x).split(";")) for m in ids] for x in b["mitigation_ids"]], dtype=int)
train, test = (b["split"] == "train").values, (b["split"] == "test").values
print(Y.sum(1).mean().round(2), "right mitigations per behaviour")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(b.loc[train, "behaviour"])
keep = Y[train].sum(0) > 0
model = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=4)).fit(vec.transform(b.loc[train, "behaviour"]), Y[train][:, keep])
scores = np.full((test.sum(), len(ids)), -1.0)
scores[:, keep] = model.predict_proba(vec.transform(b.loc[test, "behaviour"]))
top = scores.argmax(1)
print(f"trained    hit@1 {Y[test][np.arange(test.sum()), top].mean():.3f}")
popular = Y[train].sum(0).argmax()
print(f"popularity hit@1 {Y[test][:, popular].mean():.3f}  (always recommend the most common mitigation)")

**Reading it:** a few mitigations (audit, user training) apply to most techniques, so popularity is a real baseline. Skandamata gets the first recommendation right 0.792 of the time against 0.232 for popularity. It recommends; in Sati, a person or the policy engine decides.